### Monte Carlo - Euler vs Exact for Geometric Brownian Motion

Comparison of two Monte Carlo simulation approaches for pricing European options under Geometric Brownian Motion (GBM):

1. Exact terminal-value simulation
2. Euler-discretized path simulation

The goal is to study:

- discretization error
- convergence behavior
- runtime vs accuracy tradeoffs

---
Expectations:
- Unlike exact simulation, Euler introduces discretization bias that decreases as the timestep size shrinks.
- Euler simulation should converge toward the analytical benchmark as the number of timesteps increases

In [1]:
import sys
sys.path.insert(0, './../src')

In [2]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import black_scholes
from models import GeometricBrownianMotion
from models import PathGeometricBrownianMotion
from options import EuropeanCallOption
from pricers import MonteCarloPricer
from random_numbers import NumpyRNG

In [3]:
S0 = 100.0
K = 100.0
T = 1.0
r = 0.05
d = 0.02
sigma = 0.20

# number of paths and random seed
n_paths = 100_000
seed = 7

option = EuropeanCallOption(strike=K)

In [4]:
# Analytical Formula result
bs_price = black_scholes.call_price(
    S=S0,
    K=K,
    T=T,
    r=r,
    d=d,
    sigma=sigma
)

bs_price

9.227005508154036

In [5]:
# Monte Carlo for Exact GBM
exact_model = GeometricBrownianMotion(
    S0=S0,
    r=r,
    d=d,
    sigma=sigma,
    rng=NumpyRNG(seed=seed)
)

exact_mc_pricer = MonteCarloPricer(exact_model)

In [6]:
start_time = time.perf_counter()
exact_result = exact_mc_pricer.price(
    option=option,
    T=T,
    n_paths=n_paths,
    r=r
)
end_time = time.perf_counter()
exact_runtime = end_time - start_time
print(f"Monte Carlo for Exact GBM: Time taken = {exact_runtime}seconds")

Monte Carlo for Exact GBM: Time taken = 0.010994270094670355seconds


In [7]:
exact_price = exact_result["price"]
exact_std_error = exact_result["std_error"]
exact_abs_error = abs(exact_price - bs_price)

print(f"Black-Scholes Price : {bs_price:.6f}")
print(f"Exact MC Price      : {exact_price:.6f}")
print(f"Absolute Error      : {exact_abs_error:.6f}")
print(f"MC Std Error        : {exact_std_error:.6f}")
print(f"Runtime (seconds)   : {exact_runtime:.6f}")

Black-Scholes Price : 9.227006
Exact MC Price      : 9.185329
Absolute Error      : 0.041677
MC Std Error        : 0.043546
Runtime (seconds)   : 0.010994


### Euler - Path GBM Analysis

In [8]:
# Euler timestep configurations
step_counts = [1, 2, 4, 8, 16, 32, 64, 128, 256]

In [9]:
results = []

for n_steps in step_counts:
    # Euler GBM model
    euler_model = PathGeometricBrownianMotion(
        S0=S0,
        r=r,
        d=d,
        sigma=sigma,
        rng=NumpyRNG(seed=seed)
    )

    # MC pricer
    euler_pricer = MonteCarloPricer(euler_model)

    # Runtime measurement
    start_time = time.perf_counter()

    euler_result = euler_pricer.price(
        option=option,
        T=T,
        r=r,
        n_paths=n_paths,
        n_steps=n_steps
    )

    end_time = time.perf_counter()

    # Metrics
    euler_price = euler_result["price"]
    abs_error = abs(euler_price - bs_price)
    runtime = end_time - start_time

    results.append({
        "n_steps": n_steps,
        "euler_price": euler_price,
        "bs_price": bs_price,
        "abs_error": abs_error,
        "std_error": euler_result["std_error"],
        "runtime_seconds": runtime
    })

In [10]:
df = pd.DataFrame(results)

In [11]:
df

,n_steps,euler_price,bs_price,abs_error,std_error,runtime_seconds
0,1,9.067373,9.227006,0.159632,0.038055,0.006203
1,2,9.182913,9.227006,0.044093,0.040719,0.012140
2,4,9.214112,9.227006,0.012893,0.042296,0.029663
3,8,9.223986,9.227006,0.003019,0.043009,0.056346
4,16,9.180554,9.227006,0.046451,0.043150,0.115282
5,32,9.182407,9.227006,0.044599,0.043347,0.326236
6,64,9.229364,9.227006,0.002358,0.043671,0.505721
7,128,9.171775,9.227006,0.055231,0.043541,1.141244
8,256,9.119857,9.227006,0.107149,0.043336,2.275947
